## Introduction to dataset:


Dataset A contains hotel reviews from online travel platforms collected through data scraping, each record contains a review title, a rating score between 2.0 and 10.0, the review text and the guest type and also the website the it was scraped from.  The cleaned data supports two tasks which are regression (predicting the rating score given the review text) as well as sentiment classification which labels each review as either positive, negative or neutral. Preprocessing matters because the raw review text and raw data scraped is often noisy due to html/urls being retained and also missing reviews as well as duplicates could add to the noise making it more difficult for a classifier to learn the true signal of the data. Inconsistent casing and no lemmatization make for sparse features and weaker word-rating signal which could further affect a model's ability to learn the true signal of the data.



In [1]:
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)


True

In [2]:
df = pd.read_csv('Dataset_A.csv')

print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData Types:")
print(df.dtypes)

Dataset Shape: 7614 rows x 5 columns

Columns: ['title', 'rating', 'review_text', 'guest_type', 'origin_file']

Data Types:
title              str
rating         float64
review_text        str
guest_type         str
origin_file        str
dtype: object


## EDA

In [3]:
print("Sample Records:")
df.head(5)

Sample Records:


,title,rating,review_text,guest_type,origin_file
0,“Good for the price”,10.0,"Very new hotel, clean & close to MRT",Couple,hotel_review_dataset.csv
1,“Exceptional”,9.2,"Price has increased significantly post Covid, ...",Business traveler,Javian_Raw_Dataset.csv
2,“Unexpectedly beautiful”,10.0,Interiors are shockingly nice. Ambiance is rel...,Couple,Javian_Raw_Dataset.csv
3,“Convenient location”,8.8,"The room is abit small and crowded, but I will...",Couple,hotel_review_dataset.csv
4,“Clean & value for money”,9.2,Very clean and spacious room. One of the chea...,Family with teens,amara_reviews.csv


In [4]:
print("Sample Review Text (Row 0):")
print(df['review_text'].iloc[0])
print("\nSample Review Text (Row 100):")
if len(df) > 100:
    print(df['review_text'].iloc[100])

Sample Review Text (Row 0):
Very new hotel, clean & close to MRT

Sample Review Text (Row 100):
Value for money


In [5]:
original_count = len(df)
print(f"Original dataset size: {original_count} records")


Original dataset size: 7614 records


In [6]:
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pct
})
print(missing_df)

missing_text = df['review_text'].isnull().sum()
missing_rating = df['rating'].isnull().sum()
print(f"\nCritical: {missing_text} reviews have missing text")
print(f"Critical: {missing_rating} reviews have missing ratings")

             Missing Count  Missing %
title                    0       0.00
rating                   0       0.00
review_text             68       0.89
guest_type               2       0.03
origin_file              0       0.00

Critical: 68 reviews have missing text
Critical: 0 reviews have missing ratings


In [7]:
df['text_length'] = df['review_text'].fillna('').str.strip().str.len()

empty_reviews = (df['text_length'] == 0).sum()
very_short_reviews = ((df['text_length'] > 0) & (df['text_length'] < 10)).sum()

print(f"Empty reviews (0 characters): {empty_reviews}")
print(f"Very short reviews (<10 chars): {very_short_reviews}")

if very_short_reviews > 0:
    print("\nExamples of very short reviews:")
    short_mask = (df['text_length'] > 0) & (df['text_length'] < 10)
    print(df[short_mask][['review_text', 'rating']].head())

Empty reviews (0 characters): 68
Very short reviews (<10 chars): 304

Examples of very short reviews:
   review_text  rating
33   Excellent     9.6
37      Nice..     9.3
53        good    10.0
84     is good     8.4
91          ok     6.0


In [8]:
exact_duplicates = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_duplicates}")

text_duplicates = df['review_text'].duplicated().sum()
print(f"Duplicate review texts: {text_duplicates}")

dup_texts = df[df['review_text'].duplicated(keep=False)]
if len(dup_texts) > 0:
    print("\nExample: Same text with potentially different ratings:")
    sample_dup = dup_texts.groupby('review_text')['rating'].agg(['count', 'nunique', 'min', 'max'])
    sample_dup = sample_dup[sample_dup['count'] > 1].head(3)
    print(sample_dup)

Exact duplicate rows: 37
Duplicate review texts: 313

Example: Same text with potentially different ratings:
                                                    count  nunique  min   max
review_text                                                                  
-                                                       7        6  4.8   9.2
.                                                       5        3  6.4  10.0
A Front Office lady named "ATIQAH" is really ar...      2        2  5.6   6.0


In [9]:
print(f"Rating range: {df['rating'].min()} to {df['rating'].max()}")
print(f"\nRating value counts:")
rating_dist = df['rating'].value_counts().sort_index()
print(rating_dist)

Rating range: 2.0 to 10.0

Rating value counts:
rating
2.0       52
2.4       22
2.5        1
2.8       39
3.2       41
3.6       49
4.0       61
4.3        1
4.4       96
4.7        2
4.8       79
5.0        4
5.2      118
5.3        2
5.6      131
5.7        2
6.0      186
6.3        8
6.4      187
6.7       13
6.8      261
7.0       28
7.2      289
7.3       25
7.6      382
7.7       31
8.0      655
8.3       23
8.4      494
8.7       38
8.8      583
9.0       32
9.2      647
9.3       32
9.6      782
9.7       22
10.0    2196
Name: count, dtype: int64


In [10]:
html_pattern = r'<[^>]+>'
has_html = df['review_text'].fillna('').str.contains(html_pattern, regex=True).sum()
print(f"Reviews containing HTML tags: {has_html}")

url_pattern = r'http[s]?://\S+|www\.\S+'
has_urls = df['review_text'].fillna('').str.contains(url_pattern, regex=True).sum()
print(f"Reviews containing URLs: {has_urls}")

special_chars = df['review_text'].fillna('').str.count(r'[^\w\s]').mean()
print(f"Average special characters per review: {special_chars:.1f}")

Reviews containing HTML tags: 0
Reviews containing URLs: 0
Average special characters per review: 12.9


## Data cleaning

In [11]:
before_count = len(df)
df = df.dropna(subset=['review_text', 'rating'])
after_missing = len(df)
print(f"Removed {before_count - after_missing} rows with missing text/rating")

df = df.drop_duplicates()
after_dup = len(df)
print(f"Removed {after_missing - after_dup} duplicate rows")

df = df[df['review_text'].str.strip().str.len() > 0]
after_empty = len(df)
print(f"Removed {after_dup - after_empty} empty reviews")

print(f"\nClean dataset size: {len(df)} records (removed {original_count - len(df)} total)")

Removed 68 rows with missing text/rating
Removed 36 duplicate rows
Removed 0 empty reviews

Clean dataset size: 7510 records (removed 104 total)


In [12]:
print("Before lowercasing:")
print(df['review_text'].iloc[0][:100])

df['cleaned_text'] = df['review_text'].str.lower()

print("\nAfter lowercasing:")
print(df['cleaned_text'].iloc[0][:100])

Before lowercasing:
Very new hotel, clean & close to MRT

After lowercasing:
very new hotel, clean & close to mrt


In [13]:
def remove_html(text):

    text = re.sub(r'<[^>]+>', ' ', text)

    text = re.sub(r'&nbsp;', ' ', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&quot;', '"', text)
    text = re.sub(r'&#\d+;', ' ', text)
    return text

df['cleaned_text'] = df['cleaned_text'].apply(remove_html)
print("HTML tags and entities removed.")

HTML tags and entities removed.


In [14]:
boilerplate_patterns = [
    r'i stayed at this hotel',
    r'we stayed at this hotel',
    r'i booked this hotel',
    r'we booked this hotel',
    r'i stayed here',
    r'we stayed here',
    r'stayed at the hotel',
    r'booked this property',
    r'overall[,]?\s*(i|we)?\s*would',
    r'in conclusion',
]

def remove_boilerplate(text):
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    return text

df['cleaned_text'] = df['cleaned_text'].apply(remove_boilerplate)
print("Boilerplate phrases removed.")

Boilerplate phrases removed.


In [15]:
def remove_urls_emails(text):

    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    return text

df['cleaned_text'] = df['cleaned_text'].apply(remove_urls_emails)
print("URLs and email addresses removed.")

URLs and email addresses removed.


In [16]:
def remove_numbers_special(text):
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['cleaned_text'] = df['cleaned_text'].apply(remove_numbers_special)
print("Numbers and special characters removed.")

Numbers and special characters removed.


## Tokenize

In [17]:
def tokenize_text(text):
    try:
        return word_tokenize(text)
    except:
        return text.split()

df['tokens'] = df['cleaned_text'].apply(tokenize_text)

print("Sample tokenised review:")
print(df['tokens'].iloc[0][:20])

Sample tokenised review:
['very', 'new', 'hotel', 'clean', 'close', 'to', 'mrt']


In [18]:
stop_words = set(stopwords.words('english'))

domain_stopwords = {'hotel', 'stay', 'stayed', 'room', 'night', 'nights', 'day', 'days'}
stop_words = stop_words.union(domain_stopwords)

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words and len(token) > 1]

df['tokens_clean'] = df['tokens'].apply(remove_stopwords)

print(f"Stopwords in use: {len(stop_words)} words")
print(f"\nBefore stopword removal: {len(df['tokens'].iloc[0])} tokens")
print(f"After stopword removal: {len(df['tokens_clean'].iloc[0])} tokens")
print(f"\nSample cleaned tokens: {df['tokens_clean'].iloc[0][:15]}")

Stopwords in use: 206 words

Before stopword removal: 7 tokens
After stopword removal: 4 tokens

Sample cleaned tokens: ['new', 'clean', 'close', 'mrt']


In [19]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]

df['tokens_lemmatized'] = df['tokens_clean'].apply(lemmatize_tokens)

print("Sample comparison (before vs after lemmatization):")
print(f"Before: {df['tokens_clean'].iloc[0][:10]}")
print(f"After:  {df['tokens_lemmatized'].iloc[0][:10]}")

Sample comparison (before vs after lemmatization):
Before: ['new', 'clean', 'close', 'mrt']
After:  ['new', 'clean', 'close', 'mrt']


In [20]:
df['cleaned_text_final'] = df['tokens_lemmatized'].apply(lambda x: ' '.join(x))

print("Final cleaned text sample:")
print(df['cleaned_text_final'].iloc[0])

Final cleaned text sample:
new clean close mrt


## Feature Engineering


In [21]:
df['review_length'] = df['review_text'].str.len()

df['word_count'] = df['tokens_lemmatized'].apply(len)

def avg_token_length(tokens):
    if len(tokens) == 0:
        return 0
    return np.mean([len(token) for token in tokens])

df['avg_token_length'] = df['tokens_lemmatized'].apply(avg_token_length)

print("Engineered Feature Statistics:")
print(df[['review_length', 'word_count', 'avg_token_length']].describe())

Engineered Feature Statistics:
       review_length   word_count  avg_token_length
count    7510.000000  7510.000000       7510.000000
mean      217.969907    18.716778          5.397395
std       271.879854    23.788108          1.832431
min         1.000000     0.000000          0.000000
25%        53.000000     4.000000          5.125000
50%       129.000000    11.000000          5.739428
75%       274.000000    24.000000          6.263158
max      3999.000000   359.000000         27.000000


In [22]:
df['rating_bucket'] = pd.cut(df['rating'], bins=[0, 4, 6, 8, 10], labels=['Low (<=4)', 'Medium (4-6)', 'Good (6-8)', 'High (8-10)'])

feature_by_rating = df.groupby('rating_bucket')[['review_length', 'word_count', 'avg_token_length']].mean()
print(feature_by_rating.round(2))

               review_length  word_count  avg_token_length
rating_bucket                                             
Low (<=4)             411.36       34.91              5.50
Medium (4-6)          352.44       30.32              5.45
Good (6-8)            231.35       19.94              5.28
High (8-10)           184.79       15.85              5.43


In [23]:
empty_after_clean = (df['cleaned_text_final'].str.strip() == '').sum()
print(f"Empty documents after cleaning: {empty_after_clean}")

if empty_after_clean > 0:
    df = df[df['cleaned_text_final'].str.strip() != '']
    print(f"Removed {empty_after_clean} documents that became empty after cleaning")

Empty documents after cleaning: 556
Removed 556 documents that became empty after cleaning


In [24]:
print(f"Unique ratings preserved: {df['rating'].nunique()}")
print(f"Rating range: {df['rating'].min()} to {df['rating'].max()}")
print(f"All rows have valid ratings: {df['rating'].notna().all()}")

Unique ratings preserved: 37
Rating range: 2.0 to 10.0
All rows have valid ratings: True


In [25]:
sample_idx = 5

print(f"Original text (Row {sample_idx}):")

print(df['review_text'].iloc[sample_idx][:500])

print(f"\nCleaned text (Row {sample_idx}):")

print(df['cleaned_text_final'].iloc[sample_idx][:500])

Original text (Row 5):
I like this hotel.rae is good for family members to stay with room separately if kids are grwon. Access to other sightseeing place by taxi is good. Convenient stor near to hotel is very good.

Cleaned text (Row 5):
like rae good family member separately kid grwon access sightseeing place taxi good convenient stor near good


In [26]:
print("\nFinal dataset summary")
print(f"Original dataset: {original_count} rows")
print(f"Cleaned dataset: {len(df)} rows")
print(f"Rows removed: {original_count - len(df)} ({((original_count - len(df))/original_count*100):.1f}%)")

print(f"\nFinal columns: {df.columns.tolist()}")


Final dataset summary
Original dataset: 7614 rows
Cleaned dataset: 6954 rows
Rows removed: 660 (8.7%)

Final columns: ['title', 'rating', 'review_text', 'guest_type', 'origin_file', 'text_length', 'cleaned_text', 'tokens', 'tokens_clean', 'tokens_lemmatized', 'cleaned_text_final', 'review_length', 'word_count', 'avg_token_length', 'rating_bucket']


In [27]:
export_columns = [
    'title',
    'rating',
    'review_text',
    'cleaned_text_final',
    'guest_type',
    'review_length',
    'word_count',
    'avg_token_length'
]

df_export = df[export_columns].copy()
df_export = df_export.rename(columns={'cleaned_text_final': 'cleaned_text'})

output_filename = 'datasetA_cleaned.csv'
df_export.to_csv(output_filename, index=False)

print(f"Cleaned dataset exported to: {output_filename}")
print(f"   Shape: {df_export.shape[0]} rows x {df_export.shape[1]} columns")
print(f"\n   Columns: {df_export.columns.tolist()}")

Cleaned dataset exported to: datasetA_cleaned.csv
   Shape: 6954 rows x 8 columns

   Columns: ['title', 'rating', 'review_text', 'cleaned_text', 'guest_type', 'review_length', 'word_count', 'avg_token_length']
